In [48]:
import pandas as pd
import glob
import os

# --- CONFIGURATION ---
# These match the folder names you confirmed earlier
folders = {
    "Enrolment": "api_data_aadhar_enrolment",
    "Demographic": "api_data_aadhar_demographic",
    "Biometric": "api_data_aadhar_biometric"
}

def load_and_merge(folder_name, label):
    print(f"📂 Looking into folder: {folder_name}...")
    
    # Find all CSV files in that folder
    # We use "./" to say "look in the current directory"
    file_pattern = os.path.join(".", folder_name, "*.csv")
    all_files = glob.glob(file_pattern)
    
    if not all_files:
        print(f"❌ ERROR: No CSV files found in {folder_name}. Check the folder name!")
        return None

    print(f"   Found {len(all_files)} files. Merging them now...")
    
    # Read and Combine
    df_list = []
    for file in all_files:
        try:
            # low_memory=False handles the large file size better
            df = pd.read_csv(file, low_memory=False)
            df_list.append(df)
        except Exception as e:
            print(f"   ⚠️ Could not read {file}: {e}")
            
    if df_list:
        master_df = pd.concat(df_list, ignore_index=True)
        print(f"✅ {label} MERGED SUCCESS! Total Rows: {len(master_df)}")
        return master_df
    else:
        return None

# --- RUN THE PROCESS ---
df_enrol = load_and_merge(folders["Enrolment"], "Enrolment Data")
df_demo = load_and_merge(folders["Demographic"], "Demographic Data")
df_bio = load_and_merge(folders["Biometric"], "Biometric Data")

# --- SHOW ME THE COLUMNS ---
if df_enrol is not None:
    print("\n--- COLUMN NAMES CHECK ---")
    print("Enrolment Columns:", df_enrol.columns.tolist())
    print("Demographic Columns:", df_demo.columns.tolist())
    print("Biometric Columns:", df_bio.columns.tolist())

📂 Looking into folder: api_data_aadhar_enrolment...
   Found 3 files. Merging them now...
✅ Enrolment Data MERGED SUCCESS! Total Rows: 1006029
📂 Looking into folder: api_data_aadhar_demographic...
   Found 5 files. Merging them now...
✅ Demographic Data MERGED SUCCESS! Total Rows: 2071700
📂 Looking into folder: api_data_aadhar_biometric...
   Found 4 files. Merging them now...
✅ Biometric Data MERGED SUCCESS! Total Rows: 1861108

--- COLUMN NAMES CHECK ---
Enrolment Columns: ['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater']
Demographic Columns: ['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_']
Biometric Columns: ['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_']


In [49]:
# --- STEP 9: DATA CLEANING ---
print("⏳ Converting dates... (This takes a few seconds)")

# 1. Convert 'date' column to Datetime format
df_enrol['date'] = pd.to_datetime(df_enrol['date'], format='mixed')
df_demo['date'] = pd.to_datetime(df_demo['date'], format='mixed')
df_bio['date'] = pd.to_datetime(df_bio['date'], format='mixed')

# 2. Fill Missing Values with 0 (NaN -> 0)
# Because if a row has no data for 'age_0_5', it implies 0 enrolments
df_enrol.fillna(0, inplace=True)
df_demo.fillna(0, inplace=True)
df_bio.fillna(0, inplace=True)

# 3. Create a 'Total' Column for easier plotting later
df_enrol['total_enrolment'] = df_enrol['age_0_5'] + df_enrol['age_5_17'] + df_enrol['age_18_greater']
df_demo['total_demo_updates'] = df_demo['demo_age_5_17'] + df_demo['demo_age_17_']
df_bio['total_bio_updates'] = df_bio['bio_age_5_17'] + df_bio['bio_age_17_']

print("✅ Dates converted & Totals calculated!")

# --- QUICK CHECK ---
# Let's see the Top 5 States with highest activity
print("\n🏆 TOP 5 STATES (Total Enrolments):")
top_states = df_enrol.groupby('state')['total_enrolment'].sum().sort_values(ascending=False).head(5)
print(top_states)

⏳ Converting dates... (This takes a few seconds)
✅ Dates converted & Totals calculated!

🏆 TOP 5 STATES (Total Enrolments):
state
Uttar Pradesh     1018629
Bihar              609585
Madhya Pradesh     493970
West Bengal        375297
Maharashtra        369139
Name: total_enrolment, dtype: int64


In [50]:
import plotly.express as px

# --- STEP 10: COMPLIANCE GAP ANALYSIS ---
print("🚀 Analyzing District-wise Compliance...")

# 1. Aggregate Data by District & State
# We sum up all activity over the whole time period
enrol_summary = df_enrol.groupby(['state', 'district'])[['age_0_5']].sum().reset_index()
bio_summary = df_bio.groupby(['state', 'district'])[['bio_age_5_17']].sum().reset_index()

# 2. Merge them together
# We want to match the enrolment data with biometric data for the same district
gap_df = pd.merge(enrol_summary, bio_summary, on=['state', 'district'], how='inner')

# 3. Calculate "Update Ratio"
# Logic: For every 100 kids enrolled (0-5), how many biometric updates (5-17) are happening?
# A LOW number means people are forgetting to update.
gap_df['update_ratio'] = gap_df['bio_age_5_17'] / gap_df['age_0_5']

# 4. Filter for meaningful data (ignore tiny districts with < 1000 enrolments to avoid noise)
gap_df = gap_df[gap_df['age_0_5'] > 1000]

# 5. Find the "Bottom 10" (Districts that need help)
worst_districts = gap_df.sort_values(by='update_ratio', ascending=True).head(10)

print("⚠️ Top 10 Districts with Lowest Biometric Compliance:")
display(worst_districts[['state', 'district', 'update_ratio']])

# --- VISUALIZATION ---
# Create an interactive Bar Chart
fig = px.bar(
    worst_districts,
    x='district',
    y='update_ratio',
    color='state',
    title='⚠️ Priority Zones: Districts with Lowest Mandatory Biometric Updates',
    labels={'update_ratio': 'Compliance Index (Lower is Worse)', 'district': 'District'},
    text_auto='.2f'
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

🚀 Analyzing District-wise Compliance...
⚠️ Top 10 Districts with Lowest Biometric Compliance:


,state,district,update_ratio
383,Karnataka,Bengaluru Rural,0.003475
142,Bihar,Pashchim Champaran,0.007905
231,Gujarat,Banas Kantha,0.009838
256,Gujarat,Sabar Kantha,0.054478
226,Gujarat,Ahmadabad,0.233374
239,Gujarat,Dohad,0.433340
887,Uttar Pradesh,Barabanki,0.563169
29,Andhra Pradesh,Mahabub Nagar,0.761816
570,Meghalaya,East Garo Hills,0.963439
578,Meghalaya,South West Garo Hills,1.028134


In [51]:
# --- STEP 11: ANOMALY DETECTION (SPIKE FINDER) ---
print("🚨 Scanning for Suspicious Activity (Anomalies)...")

# 1. Group by District and Month to see trends
# We use 'M' to group data by Month
monthly_activity = df_demo.set_index('date').groupby('district').resample('M')['total_demo_updates'].sum().reset_index()

# 2. Calculate Statistics per District
# Mean = Average monthly updates
# Std = How much it usually fluctuates
stats = monthly_activity.groupby('district')['total_demo_updates'].agg(['mean', 'std']).reset_index()

# 3. Merge stats back to the main data
monthly_activity = pd.merge(monthly_activity, stats, on='district')

# 4. Calculate Z-Score
# Formula: (Current Value - Average) / Fluctuation
# If Z-Score > 3, it means the value is HUGE (3 times standard deviation) - A STATISTICAL ANOMALY
monthly_activity['z_score'] = (monthly_activity['total_demo_updates'] - monthly_activity['mean']) / monthly_activity['std']

# 5. Filter for Extreme Anomalies (Z-Score > 3)
anomalies = monthly_activity[monthly_activity['z_score'] > 3].sort_values(by='z_score', ascending=False)

print("⚠️ Top 5 Suspicious Spikes Detected:")
display(anomalies.head(5))

# --- VISUALIZATION OF THE #1 ANOMALY ---
# Let's plot the weirdest district to see the spike
if not anomalies.empty:
    top_anomaly_district = anomalies.iloc[0]['district']
    print(f"\n📈 Plotting the suspicious trend for: {top_anomaly_district}")
    
    district_data = monthly_activity[monthly_activity['district'] == top_anomaly_district]
    
    fig = px.line(
        district_data, 
        x='date', 
        y='total_demo_updates',
        title=f'🚨 Anomaly Detection: Suspicious Spike in {top_anomaly_district}',
        markers=True
    )
    # Add a red line for the "Threshold"
    threshold = district_data['mean'].iloc[0] + (3 * district_data['std'].iloc[0])
    fig.add_hline(y=threshold, line_dash="dash", line_color="red", annotation_text="Anomaly Threshold")
    
    fig.show()

🚨 Scanning for Suspicious Activity (Anomalies)...


C:\Users\swara\AppData\Local\Temp\ipykernel_2460\3066809944.py:6: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



⚠️ Top 5 Suspicious Spikes Detected:


,district,date,total_demo_updates,mean,std,z_score
2913,East Delhi,2025-01-31,145128,15588.416667,40908.816092,3.166544
1637,Budgam,2025-01-31,14444,1551.833333,4071.957377,3.166086
7517,North Delhi,2025-01-31,67329,7241.666667,18985.090640,3.164975
7553,North East Delhi,2025-01-31,105294,11762.333333,29566.231722,3.163463
2791,Dindori,2025-01-31,36057,3951.416667,10148.912843,3.163450



📈 Plotting the suspicious trend for: East Delhi


In [52]:
# --- STEP 12: PREDICTIVE ANALYTICS (FORECASTING) ---
print("🔮 Predicting Future Demand for Next 3 Months...")

# 1. Prepare Data: Total Enrolments per Month across ALL India
# We want to see the national trend
monthly_trend = df_enrol.set_index('date').resample('ME')['total_enrolment'].sum().reset_index()

# 2. Calculate "Moving Average" (Smooths out the noise)
# We take the average of the last 3 months to predict the next one
monthly_trend['3_month_moving_avg'] = monthly_trend['total_enrolment'].rolling(window=3).mean()

# 3. Predict the Next Month (Simple Logic: It will likely be close to the recent average)
last_avg = monthly_trend.iloc[-1]['3_month_moving_avg']
print(f"📊 Forecasted Demand for Next Month: {int(last_avg):,} enrolments")

# --- VISUALIZATION ---
import plotly.graph_objects as go

fig = go.Figure()

# Plot Actual Data
fig.add_trace(go.Scatter(
    x=monthly_trend['date'], 
    y=monthly_trend['total_enrolment'],
    mode='lines+markers',
    name='Actual Enrolments',
    line=dict(color='blue')
))

# Plot the Trend Line (Moving Average)
fig.add_trace(go.Scatter(
    x=monthly_trend['date'], 
    y=monthly_trend['3_month_moving_avg'],
    mode='lines',
    name='Trend (3-Month Avg)',
    line=dict(color='orange', dash='dash')
))

fig.update_layout(title='🔮 National Enrolment Demand Forecast', xaxis_title='Date', yaxis_title='Total Enrolments')
fig.show()

🔮 Predicting Future Demand for Next 3 Months...
📊 Forecasted Demand for Next Month: 831,124 enrolments


In [53]:
import plotly.express as px 
import pandas as pd

# --- STEP 14: THE "PROJECT DRISHTI" ALGORITHM (Occupational Burnout Detection) ---
print("👁️ Initiating Project Drishti: Scanning for Biometric Distress Zones...")

# 1. Prepare the Data
# FIX: We use the exact column names from your data
# Enrolment Adult Column: 'age_18_greater'
# Biometric Adult Column: 'bio_age_17_'
adult_enrol = df_enrol.groupby(['state', 'district'])[['age_18_greater']].sum().reset_index()
adult_bio   = df_bio.groupby(['state', 'district'])[['bio_age_17_']].sum().reset_index()

# 2. Rename columns to make them match our logic (for easier math)
adult_enrol.rename(columns={'age_18_greater': 'enrol_18_plus'}, inplace=True)
adult_bio.rename(columns={'bio_age_17_': 'bio_18_plus'}, inplace=True)

# 3. Merge Data
burnout_df = pd.merge(adult_enrol, adult_bio, on=['state', 'district'])

# 4. Calculate "Distress Ratio"
# Logic: High biometric updates / Total Enrolled Adults = Distress
burnout_df['distress_ratio'] = (burnout_df['bio_18_plus'] / burnout_df['enrol_18_plus']) * 100

# 5. Filter Noise (Ignore tiny districts with < 5000 people)
burnout_df = burnout_df[burnout_df['enrol_18_plus'] > 5000]

# 6. Find the "Red Zones" (Top 10 Districts with highest failure/update rates)
red_zones = burnout_df.sort_values(by='distress_ratio', ascending=False).head(10)

print("⚠️ RED ZONES IDENTIFIED (High Biometric Failure Probability):")
display(red_zones)

# --- VISUALIZATION ---
fig = px.scatter(
    red_zones,
    x='district',
    y='distress_ratio',
    size='enrol_18_plus', # Bubble size = Population
    color='state',
    title='🔥 Project Drishti: Occupational Biometric Burnout Zones',
    labels={'distress_ratio': 'Biometric Distress Score (%)', 'district': 'District'},
    hover_data=['enrol_18_plus', 'bio_18_plus']
)

# Add a "Threshold Line" to show danger zone
fig.add_hline(y=red_zones['distress_ratio'].mean(), line_dash="dash", line_color="red", annotation_text="Critical Failure Threshold")

fig.show()

👁️ Initiating Project Drishti: Scanning for Biometric Distress Zones...
⚠️ RED ZONES IDENTIFIED (High Biometric Failure Probability):


,state,district,enrol_18_plus,bio_18_plus,distress_ratio
572,Meghalaya,East Khasi Hills,9948,12022,120.848412
582,Meghalaya,West Khasi Hills,5310,2728,51.374765


In [54]:
import plotly.express as px

# ==========================================
# RE-GENERATING GRAPH 1: MIGRATION MAGNETS (Horizontal)
# ==========================================
# Sort data so the biggest bar is at the top
top_magnets_sorted = top_magnets.sort_values(by='migration_index', ascending=True)

fig_mig = px.bar(
    top_magnets_sorted,
    x='migration_index', 
    y='district', 
    orientation='h',  # <--- This makes it Horizontal
    color='migration_index',
    title='🏙️ The "Migration Magnet" Index: Top Influx Districts',
    labels={'migration_index': 'Migration Intensity Score', 'district': 'District'},
    text_auto='.1f',
    color_continuous_scale='Viridis' # Professional Green-Blue-Yellow Gradient
)

# Clean up the layout
fig_mig.update_layout(yaxis_title=None, showlegend=False)
fig_mig.show()

In [55]:
import plotly.express as px
import pandas as pd# 1. Filter for Student Age Group (5-17) in Biometric Updates
# We want to see WHEN they update
student_updates = df_bio[['date', 'bio_age_5_17']].copy()

# 2. Extract Month
student_updates['month_name'] = student_updates['date'].dt.month_name()
student_updates['month_num'] = student_updates['date'].dt.month

# 3. Aggregate by Month (Summing up all years to see the pattern)
monthly_pattern = student_updates.groupby(['month_num', 'month_name'])['bio_age_5_17'].sum().reset_index()

# 4. Sort by Calendar Month (Jan to Dec)
monthly_pattern = monthly_pattern.sort_values('month_num')

print("\n🎒 Monthly Student Update Pattern:")
display(monthly_pattern)

# --- VISUALIZATION 2: SEASONALITY LINE CHART ---
fig_season = px.line(
    monthly_pattern,
    x='month_name',
    y='bio_age_5_17',
    title='🎒 The "Student Surge": Identifying the Server Crash Season',
    markers=True,
    labels={'bio_age_5_17': 'Student Biometric Updates', 'month_name': 'Month'}
)

# Highlight the Peak
max_val = monthly_pattern['bio_age_5_17'].max()
fig_season.add_hline(y=max_val, line_dash="dash", line_color="red", annotation_text="System Overload Peak")

fig_season.show()


🎒 Monthly Student Update Pattern:


,month_num,month_name,bio_age_5_17
0,1,January,20827015
1,2,February,666600
2,3,March,563950
3,4,April,512443
4,5,May,389715
5,6,June,533374
6,7,July,315285
7,8,August,552308
8,9,September,2286437
9,10,October,2787083


In [56]:
import plotly.express as px
import pandas as pd

print("🌍 Initiating Geopolitical & Climate Analysis modules...")

# Logic: Check if Enrolments drop drastically during Monsoon (June-Sept) in flood-prone states
target_state = 'Assam' # Or Bihar/Kerala
climate_df = df_enrol[df_enrol['state'] == target_state].copy()

# Group by Month to see the seasonal trend
climate_df['month_name'] = climate_df['date'].dt.month_name()
climate_df['month_num'] = climate_df['date'].dt.month
flood_impact = climate_df.groupby(['month_num', 'month_name'])['total_enrolment'].sum().reset_index().sort_values('month_num')

print(f"\n🌪️ Analyzing Climate Impact in {target_state}:")
display(flood_impact)

# Visualization
fig_climate = px.line(
    flood_impact,
    x='month_name', 
    y='total_enrolment',
    title=f'🌪️ Climate Impact Monitor: Enrolment Dip during Floods ({target_state})',
    markers=True,
    labels={'total_enrolment': 'New Aadhaar Enrolments'}
)
# Highlight Monsoon
fig_climate.add_vrect(x0="June", x1="September", fillcolor="blue", opacity=0.1, annotation_text="Monsoon/Flood Season")
fig_climate.show()

🌍 Initiating Geopolitical & Climate Analysis modules...

🌪️ Analyzing Climate Impact in Assam:


,month_num,month_name,total_enrolment
0,1,January,100641
1,2,February,6112
2,3,March,6168
3,4,April,2894
4,5,May,4876
5,6,June,4425
6,7,July,1793
7,8,August,5013
8,9,September,22369
9,10,October,31950


In [57]:
# ==========================================
# RE-GENERATING GRAPH 2: TRADE WAR MIGRATION (Horizontal)
# ==========================================
# Sort data so the biggest bar is at the top
top_hubs_sorted = top_hubs.sort_values(by='migration_index', ascending=True)

fig_trade = px.bar(
    top_hubs_sorted,
    x='migration_index',
    y='district',
    orientation='h', # <--- This makes it Horizontal
    color='state',   # Color by State to show the regional cluster
    title='🏭 Make in India 2.0: Labor Surge in Industrial Hubs',
    labels={'migration_index': 'Labor Influx Intensity', 'district': 'Industrial Hub'},
    text_auto='.1f',
    color_discrete_sequence=px.colors.qualitative.Bold # distinct bold colors
)

# Clean up the layout
fig_trade.update_layout(yaxis_title=None)
fig_trade.show()

In [58]:
import plotly.express as px
import pandas as pd

print("🚀 Generating Operational Insights (The Partner Integration)...")

# ==========================================
# 1. THE "SATURDAY PANIC" (Day of Week Analysis)
# ==========================================
# We analyze WHEN people are visiting centers
# Ensure 'date' is datetime
df_enrol['date'] = pd.to_datetime(df_enrol['date'])
df_enrol['day_name'] = df_enrol['date'].dt.day_name()

# Order of days for the plot
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Group by Day
daily_load = df_enrol.groupby('day_name')['total_enrolment'].sum().reindex(day_order).reset_index()

print("\n📅 Weekly Workload Distribution:")
display(daily_load)

fig_saturday = px.bar(
    daily_load,
    x='day_name',
    y='total_enrolment',
    title='📅 The "Saturday Panic": Weekly Operational Bottleneck',
    color='total_enrolment',
    color_continuous_scale='Reds', # Red shows the danger zone (Saturday)
    labels={'total_enrolment': 'Transaction Volume', 'day_name': 'Day of Week'}
)
fig_saturday.show()

🚀 Generating Operational Insights (The Partner Integration)...

📅 Weekly Workload Distribution:


,day_name,total_enrolment
0,Monday,883849
1,Tuesday,1292216
2,Wednesday,618175
3,Thursday,607377
4,Friday,486732
5,Saturday,827131
6,Sunday,720222


In [59]:
# ==========================================
# 2. THE "SUPER-CENTERS" (Pincode Level Hotspots)
# ==========================================
# Finding the specific buildings that are overcrowded
# Assuming you have a 'pincode' column. If not, use 'district' again or skip this specific graph.
# If pincode exists:
if 'pincode' in df_enrol.columns:
    pincode_load = df_enrol.groupby(['state', 'district', 'pincode'])['total_enrolment'].sum().reset_index()
    top_pincodes = pincode_load.sort_values(by='total_enrolment', ascending=False).head(10)

    # Convert pincode to string for plotting
    top_pincodes['pincode'] = top_pincodes['pincode'].astype(str)

    print("\n📍 Top 10 Busiest Pincodes (Super-Centers):")
    display(top_pincodes)

    fig_pincodes = px.bar(
        top_pincodes,
        x='pincode',
        y='total_enrolment',
        color='state',
        title='📍 The "Super-Centers": Top 10 Most Overcrowded Locations',
        labels={'total_enrolment': 'Footfall Volume', 'pincode': 'Center Pincode'},
        text_auto='.2s'
    )
    fig_pincodes.show()
else:
    print("Pincode column not found, skipping Super-Center graph.")


📍 Top 10 Busiest Pincodes (Super-Centers):


,state,district,pincode,total_enrolment
25639,Uttar Pradesh,Moradabad,244001,15122
23996,Uttar Pradesh,Aligarh,202001,11833
15955,Meghalaya,West Khasi Hills,793119,11321
5650,Delhi,West Delhi,110059,10462
25836,Uttar Pradesh,Saharanpur,247001,10189
25829,Uttar Pradesh,Rampur,244901,9572
25591,Uttar Pradesh,Meerut,250002,9378
23967,Uttar Pradesh,Agra,282001,8686
13886,Maharashtra,Aurangabad,431001,8645
25929,Uttar Pradesh,Shahjahanpur,242001,8511


In [60]:
import plotly.express as px
import pandas as pd

print("🚀 Generating Insight 4: The Compliance Index...")

# ==========================================
# INSIGHT 4: THE COMPLIANCE INDEX (Child vs. Adult Updates)
# ==========================================

# 1. Aggregate Biometric Updates by State
# bio_age_5_17 = Mandatory Child Updates (Good compliance)
# bio_age_17_  = Adult Updates (Corrections/Burnout)
compliance_df = df_bio.groupby('state')[['bio_age_5_17', 'bio_age_17_']].sum().reset_index()

# 2. Rename for clarity
compliance_df.rename(columns={
    'bio_age_5_17': 'Mandatory Child Updates', 
    'bio_age_17_': 'Adult Corrections'
}, inplace=True)

# 3. Calculate "Compliance Score" (% of work that is Child Updates)
# High % means the state is successfully focusing on mandatory updates.
compliance_df['Total'] = compliance_df['Mandatory Child Updates'] + compliance_df['Adult Corrections']
compliance_df['Compliance_Score'] = (compliance_df['Mandatory Child Updates'] / compliance_df['Total']) * 100

# 4. Sort and Filter (Top 15 States for readability)
compliance_df = compliance_df.sort_values(by='Total', ascending=False).head(15)

print("\n🏆 Top States by Biometric Activity:")
display(compliance_df)

# 5. Create Stacked Bar Chart
fig_comp = px.bar(
    compliance_df,
    x=['Mandatory Child Updates', 'Adult Corrections'],
    y='state',
    orientation='h', # Horizontal for readability
    title='🏆 The Compliance Index: Which States are Driving Mandatory Updates?',
    labels={'value': 'Biometric Update Volume', 'variable': 'Update Type', 'state': 'State'},
    color_discrete_map={
        'Mandatory Child Updates': '#2ca02c', # Green (Good)
        'Adult Corrections': '#d62728'        # Red (Correction/Error)
    }
)

fig_comp.update_layout(yaxis_title=None, legend_title_text='Update Category')
fig_comp.show()

🚀 Generating Insight 4: The Compliance Index...

🏆 Top States by Biometric Activity:


,state,Mandatory Child Updates,Adult Corrections,Total,Compliance_Score
44,Uttar Pradesh,6207105,3370630,9577735,64.807650
27,Maharashtra,3512712,5713427,9226139,38.073478
26,Madhya Pradesh,3200117,2723654,5923771,54.021619
5,Bihar,2208141,2689446,4897587,45.086305
40,Tamil Nadu,2227252,2470865,4698117,47.407334
38,Rajasthan,2066747,1928208,3994955,51.733924
2,Andhra Pradesh,2241448,1473144,3714592,60.341701
16,Gujarat,1460655,1735859,3196514,45.695248
8,Chhattisgarh,884553,1764176,2648729,33.395376
22,Karnataka,1244999,1390955,2635954,47.231439
